# Production regression replication

This notebook estimates the paper-aligned production OLS specification from the 67-row aggregate table and computes HC3 robust inference. Benchmark comparisons are calculated in the notebook rather than loaded from generated CSV files.

In [1]:
from pathlib import Path
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from broadway_research.ingest import load

# Public clones must supply the authorized six-sheet workbook separately.
# Set BROADWAY_INPUT_DIR to the directory containing that workbook.
input_value = os.environ.get("BROADWAY_INPUT_DIR")
if not input_value:
    raise FileNotFoundError("Set BROADWAY_INPUT_DIR to the authorized workbook directory.")
input_dir = Path(input_value)
if not input_dir.exists() or not list(input_dir.glob("*.xlsx")):
    raise FileNotFoundError(
        "Authorized source workbook not found. Set BROADWAY_INPUT_DIR to its containing directory. "
        "The workbook is intentionally excluded from the public repository."
    )

production, market = load(input_dir)
print(f"Loaded authorized inputs: {production.shape[0]} production aggregates and {market.shape[0]} market weeks")

Loaded authorized inputs: 67 production aggregates and 501 market weeks


In [2]:
from broadway_research.production_analysis import formula, fit

model_formula = formula(production)
model_formula

'log_gross ~ log_seats + Adapted + Revival + C(Year) + C(Music_Style) + Plot_Comedy + Plot_Tragedy + Plot_Romance + Plot_Mystery + Plot_Drama + Plot_Adventure + Plot_Crime + Plot_Biographical + Plot_Fantasy'

In [3]:
fit_hc3 = fit(production)
ci = fit_hc3.conf_int()
key_results = pd.DataFrame({
    "estimate": fit_hc3.params,
    "hc3_std_error": fit_hc3.bse,
    "hc3_p_value": fit_hc3.pvalues,
    "ci95_low": ci[0],
    "ci95_high": ci[1],
})
key_results.loc[["log_seats", "Adapted", "Revival"]]

,estimate,hc3_std_error,hc3_p_value,ci95_low,ci95_high
log_seats,1.007385,0.243747,0.000036,0.529649,1.485121
Adapted,0.286721,0.173605,0.098622,-0.053538,0.626980
Revival,-0.284169,0.155193,0.067090,-0.588343,0.020004


In [4]:
benchmarks = {
    "n": 67,
    "r_squared": 0.6079569056109908,
    "adjusted_r_squared": 0.42500346156278657,
    "log_seats_coefficient": 1.0073848481499095,
    "log_seats_hc3_p": 3.582017220037733e-05,
    "adapted_coefficient": 0.2867209169,
}
computed = {
    "n": int(fit_hc3.nobs),
    "r_squared": fit_hc3.rsquared,
    "adjusted_r_squared": fit_hc3.rsquared_adj,
    "log_seats_coefficient": fit_hc3.params["log_seats"],
    "log_seats_hc3_p": fit_hc3.pvalues["log_seats"],
    "adapted_coefficient": fit_hc3.params["Adapted"],
}
comparison = pd.DataFrame({"benchmark": benchmarks, "computed": computed})
comparison["absolute_difference"] = (comparison["computed"] - comparison["benchmark"]).abs()
comparison["within_tolerance"] = comparison["absolute_difference"] <= 1e-6
assert comparison["within_tolerance"].all()
comparison

,benchmark,computed,absolute_difference,within_tolerance
n,67.000000,67.000000,0.000000e+00,True
r_squared,0.607957,0.607957,6.661338e-16,True
adjusted_r_squared,0.425003,0.425003,9.992007e-16,True
log_seats_coefficient,1.007385,1.007385,1.776357e-15,True
log_seats_hc3_p,0.000036,0.000036,2.419126e-18,True
adapted_coefficient,0.286721,0.286721,1.402339e-11,True


In [5]:
pd.Series({
    "N": int(fit_hc3.nobs),
    "R_squared": fit_hc3.rsquared,
    "adjusted_R_squared": fit_hc3.rsquared_adj,
    "design_columns": fit_hc3.model.exog.shape[1],
})

N                     67.000000
R_squared              0.607957
adjusted_R_squared     0.425003
design_columns        22.000000
dtype: float64

Because gross and capacity enter in natural logs, the capacity coefficient is an approximate elasticity: conditional on included controls, a 1% capacity difference is associated with about a 1.01% gross difference. This is not a causal estimate of adding seats; productions may be assigned theatres partly in response to expected demand.